# 02 · EDA: the population, the target and the features that matter

**What it does.** A fast picture of one use case: who is in the population, how often the target
happens and which features order clients by it. EDA changes nothing: it drops no row or feature
and records no decision.

**How it works.** One period, the **snapshot** (by default the window's last), is collected whole
into pandas; the other window periods are only counted in Spark. On the synthetic data (100,000
clients) it takes about 3 minutes on a laptop.

**Concordance**, the number the ranking is built on, says how well a feature alone orders
clients by the target: 0.5 is chance, 1 a perfect ordering. For a yes/no target it is the
feature's AUC; step 3's notes say more.

**Before you start.** Run `01_target_factory` first: EDA reads the model input it saves. For the
other example, change `CONFIG` in the setup cell. The CLI: `python -m core_modelling run
conf/base/cc_acquisition.yml --stages eda`.

**Automation levels** (`automation_level` in the config; this example is `assisted`): at
`autonomous` EDA runs by itself before Feature Checks and nobody needs to read it; at `assisted`
you read it here and record your decisions (step 9) before Feature Checks.

**What it writes**, under `<roots.data>/<use case>/04_feature/<data-definition hash>/eda/`:
`report.html`, `eda.json` (settings, counts, key features, warnings, timings), `tables/` and
`resolved_config.yml`; one folder up, the feature ledger (`ledger.parquet`, one row per
candidate). A rerun replaces `eda/` and removes Feature Checks' folder, which is then out of date.

**Docs:** `docs/components/eda.md` · `docs/components/reporting.md` · next: `03_feature_checks`.

In [ ]:
from pathlib import Path

from IPython.display import HTML, display

import core_modelling
from core_modelling.config import load_config
from core_modelling.config.audit_log import show_decisions
from core_modelling.eda import compare_sides, lift_by_group, pivot_target, rate_by_period, run_eda
from core_modelling.environment import get_spark_session, load_environment

# The use case and where the framework runs, as absolute paths. The repo's examples are found
# from the installed package; a use-case repo writes its own config path here and may read the
# environment file from $CORE_MODELLING_ENVIRONMENT instead.
REPO = Path(core_modelling.__file__).resolve().parents[2]
CONFIG = REPO / "conf/base/cc_acquisition.yml"  # or conf/base/td_acquisition.yml
ENTITY = "entity_a"
ENVIRONMENT = str(REPO / "conf/local/environment.yml")  # or None: $CORE_MODELLING_ENVIRONMENT

env = load_environment(ENVIRONMENT)
spark = get_spark_session(env)
cfg = load_config(CONFIG)
print(cfg.use_case, cfg.task, cfg.automation_level)
print(cfg.time)

## 1. Run EDA
`run_eda` applies the decision audit log to the config, reads the model input, computes every
number, writes `eda/` and returns `result`; `result.views` holds the report's tables and charts.
It prints:
- `eligibility`: an analysis filter's rows in → out (dropped before EDA); a modelling filter's
  rows tagged 1 (modelled; EDA keeps the rest as a reference group).
- `contract`: checks on the whole population, such as the eligible rows per period.
- `compared periods`: the window periods, which carry labels, and the later periods compared
  without labels. The validation period is never read.
- `collect snapshot`: its size against the collect budget. Above it, EDA warns and collects
  anyway: the population is never sampled.

**When it fails**, the message names the config field to fix or says to run notebook 01 first.
Fix the config or record a `set` (step 9) and rerun this cell; a failed run leaves no
half-written `eda/`.

In [ ]:
result = run_eda(cfg, ENTITY, env, spark)
views = result.views

## 2. The choices
What was decided, before any number: **Value** is this run's setting, **Alternatives** the other
options (`conf/base/TEMPLATE.yml` says when each fits). Check that each value is what you meant,
above all the task, decision and metric (fixed before any result is seen), the windows, the
eligibility steps and EDA's own settings (the snapshot, `eda.bins`, the key-feature rule, the
strata). The last audit-log entry applied and the two hashes identify this run.

**Decide:** EDA's settings and the strata change with a `set` (step 9); the window, eligibility
and candidates only in a new config.

In [ ]:
views["choices"]

## 3. How to read, the headline and the warnings
**How to read**: five notes on the words every later view uses. Read them once first.

**The headline**:
- `eligible rows` (= `clients`: one row per client and period) and `eligible share`: the snapshot
  rows that pass every eligibility step.
- `snapshot rows`, `snapshot positives`, `snapshot rate`: the eligible rows with a known target.
  That rate is the **base rate** every lift is measured against; `snapshot low` and `high` are
  its 95% interval.
- `window …`: the same over the whole window. `window positive events` counts distinct clients;
  `rows per event` lies between 1 and `w`, since a client is positive in each of the `w` periods
  before taking the product.

Shares show as fractions (0.031 is 3.1%). An amount target shows its mean, median, p10, p90 and
zero share instead of the rate.

**The warnings** show only those that apply, each saying what it means. What to do:
- **A thin window period**: read the trend from the other periods.
- **A thin stratum level**: don't read its rate; if most of a stratum is thin, choose other strata
  (step 9).
- **Nulls in the target column**: if a null means a value, have it filled upstream.
- **The snapshot above the collect budget**: raise `collect_budget_gb` in the environment file, or
  narrow the population in a new config.
- **Few events**, and **the labels warning** (always shown): nothing to do; the model results
  report states the latter.

In [ ]:
display(*[HTML(note) for note in views["how_to_read"]])
display(HTML(views["headline"]), HTML(views["warnings"]))

## 4. The population and the target
**The funnel**, per window period: `spine` is the rows the first eligibility step met,
`after_<filter>` the rows left after each analysis filter, `tagged_<filter>` the rows each
modelling filter keeps, `eligible` the rows every later stage models. Healthy: steady columns. A
filter that drops nothing or almost everything usually has a wrong condition.

**The target per period** (table, then chart): `rows` (eligible, target known), `support` (positive
events; the `all` row counts each client once) and the `estimate` (the rate) with its interval.
Healthy: a steady rate with overlapping intervals. A trend or a jump has a cause (a campaign, a
product or data change): find it before modelling. The `all` row's interval is wide because the
same clients repeat across periods.

**The target by stratum** (`segment` and `onboarding_channel` here), on the snapshot: the largest
`strata.max_levels` levels, the rest `__other__`. For an amount target the estimate is the mean,
with the non-zero share beside it.

**Decide:** a wrong filter needs a new config; a mostly thin stratum, or a missing split the
business uses, is a `set` of `strata.variables` (step 9).

In [ ]:
display(views["funnel"], *views["target_per_period"])
for variable, blocks in views["target_by_stratum"].items():
    display(HTML(f"<h3>Target by {variable}</h3>"), *blocks)

## 5. The ranking
Every supported candidate, best first. The numbers are the snapshot's; a `_window` column pools
all window periods (more events, less noise).
- `concordance` close to 1 is too good to be true: ask whether the feature is known before the
  target.
- `concordance_plain`: filled for a **non-monotone** feature, whose rate rises then falls (say,
  highest at middle ages). Its `concordance` puts its groups in rate order; `concordance_plain`
  keeps the feature's own order.
- `direction`: `+` when higher values go with a higher target, `-` with a lower one, empty when too
  close to 0.5 to tell. A categorical's is `+` or empty (its levels are put in rate order).
- `concordance_window` far from `concordance`: the relation changes over time or the snapshot is
  unusual; step 6's per-period chart shows which.
- `null_concordance`: how well "null or not" alone orders clients. Well above 0.5, missingness
  itself predicts: fine when its reason is known, a leak when the field is filled only after the
  event.
- `zero_share` is among non-null values; `dominant_share` (the commonest value's share) near 1
  marks an almost constant feature.
- `key_feature`: `top_n` (the top 10) or `listed`. `represented_by`: the key feature a candidate
  was skipped for, as it carries nearly the same information; it stays a candidate.

Below, **excluded automatically**: the lineage columns and the unsupported types, never ranked.

**Decide:** a feature too good to be true, or not known at scoring time, is an `exclude`; one you
want deep-dived is a `set` of `eda.key_features.include` (step 9).

In [ ]:
display(views["ranking"], views["excluded"])

## 6. The key features
One section per key feature, in rank order (the top 10 without near-duplicates, plus any listed):
- **A group table** first, for a categorical or a numeric with few values.
- **Lift by group**: a numeric's values merged into about ten groups (`[374, 419)` includes 374,
  not 419; `= 0` is one value), `null` last; the number on a bar is its positive events. Look for
  a clear shape (rising, falling, U); a lone tall bar with a wide interval is noise.
- **By stratum**: one line per stratum level. The same shape in every line: the feature works in
  every segment; in one line only: much of it may be the segment itself.
- **Boxes by target class** (numerics): boxes apart, the feature separates the classes.
- **Concordance per window period**: flat is stable; a trend is drift, which Feature Checks
  judges; below 0.5, the direction flipped.
- **One pivot table per stratum**: the numbers behind the stratum's lines.

An amount target's lift is the group mean over the overall mean; its group table always shows, and
a last chart shows the target per group.

**Decide:** groups that make no sense (a code read as a number) call for a kind override; a
feature not known at scoring time is an `exclude` (step 9).

In [ ]:
for feature, blocks in views["deep_dives"].items():
    display(HTML(f"<h3>{feature}</h3>"), *blocks)

## 7. Your own views
Four functions answer what the report leaves open. They read `result` only, so they are fast and
change nothing, and they group and grey by the report's rules. Any snapshot column works
(`rate_by_period` needs a candidate). Anything worth keeping goes into the config or the audit
log.

`pivot_target(result, rows, columns=None, where=None)`: how does the target vary by any columns,
here employment status by segment among clients aged 30 or more? `where` is a `DataFrame.query`
filter; a column name with spaces goes in backticks.

In [ ]:
pivot_target(result, "employment_status", columns="segment", where="age >= 30")

`lift_by_group(result, feature, by=None)`: does income's shape hold within each employment
status, not only within the strata?

In [ ]:
lift_by_group(result, "income", by="employment_status")

`compare_sides(result, feature, tag)`: how do the clients a modelling filter keeps out (here the
current card holders, `non_holders` = 0) differ in income from the population that is modelled?

In [ ]:
compare_sides(result, "income", "non_holders")

`rate_by_period(result, feature)`: does each income group's target rate hold in every window
period, or does one group drift?

In [ ]:
rate_by_period(result, "income")

## 8. The report file
`report.html` (about 6 MB, no internet needed) holds every view above; a rerun regenerates it.
Open the path the next cell prints in a browser (in JupyterLab: from the file browser, then
**Trust HTML**). Only the report holds:
- **Reference groups**, one per modelling filter (here the card holders `non_holders` keeps out):
  per period the rows kept and kept out, then each side's size, stratum shares and key features on
  the snapshot. SMD is the gap in means in standard deviations. Check that the filter keeps out
  the clients you meant.
- **The correlation overview** of the key features and those they represent (hover for the
  measure): 0 is no association, 1 the same information. It is the only sampled number (a seeded
  50,000 snapshot rows).
- **The appendix**: each column's kind, the feature ledger, the resolved config, the audit log's
  decisions, the collect log and the timings.

In [ ]:
REPORT = result.paths["report.html"]
print(REPORT)

## 9. What to decide and how to record it
A decision is an entry in the decision audit log: append-only, with a reason, applied to every
later run's config. At `assisted`, record any of those below before Feature Checks; at
`autonomous`, only when something is wrong.

Paste a call into your own cell after `from core_modelling.config.audit_log import
record_decision` (the setup cell defines `CONFIG`, `ENTITY` and `ENVIRONMENT`). A `set` replaces
the whole value at its path, so list what should stay too; rerun step 1 to see it.

**A wrong kind**: a code read as a number (text that holds numbers is converted upstream).
```python
record_decision(CONFIG, ENTITY, set={"features.kinds": {"<column>": "categorical"}},
                reason="A branch code: its numbers have no order", env_path=ENVIRONMENT)
```

**A different snapshot**, when the last period is unusual:
```python
record_decision(CONFIG, ENTITY, set={"eda.snapshot": "2025-06-01"},
                reason="August held the core system migration; June is a normal month",
                env_path=ENVIRONMENT)
```

**A feature to always deep-dive** (at `autonomous` the key features are the top 10 alone):
```python
record_decision(CONFIG, ENTITY, set={"eda.key_features.include": ["bureau_score"]},
                reason="The product team asks how the bureau score relates to take-up",
                env_path=ENVIRONMENT)
```

**Other strata**, when a stratum is mostly thin or a split the business uses is missing:
```python
record_decision(CONFIG, ENTITY,
                set={"strata.variables": ["segment", "onboarding_channel", "employment_status"]},
                reason="Take-up is managed by employment status", env_path=ENVIRONMENT)
```

**A feature that must not be used** (not known at scoring time, or not allowed): it stays in the
ledger and the reports, and Training leaves it out. It applies from Feature Checks on, so EDA needs
no rerun.
```python
record_decision(CONFIG, ENTITY, exclude=["cc_preapproval_score_trap_leak"],
                reason="Set after the offer, so not known at scoring time",
                evidence=["EDA report: Univariate ranking"], env_path=ENVIRONMENT)
```

`eda.bins`, `eda.key_features.top_n` and `max_correlation`, and `strata.max_levels` and
`min_events` change the same way. The next cell lists the log's entries.

In [ ]:
show_decisions(cfg, ENTITY, env)

**Next: `03_feature_checks`.** It reads EDA's artifacts (rerun step 1 after a `set`) and screens
every candidate for quality, leakage, time alignment, stability and redundancy; a flagged feature
stays out of training until a person keeps or excludes it.